In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier

In [2]:
df = pd.read_csv("Hotel Reservations.csv")
df = df.drop(columns = ["Booking_ID"])

label_encoders = {}

for col in df.select_dtypes(include = ["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

x = df.drop(columns = ["booking_status"])
y = df["booking_status"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
#################
# RANDOM FOREST #
#################

rf_model = RandomForestClassifier(n_estimators = 200, max_depth = None, min_samples_split = 2, min_samples_leaf = 1, random_state = 42)
rf_model.fit(x_train, y_train)
y_pred = rf_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# importances = pd.Series(rf_model.feature_importances_, index = x.columns).sort_values(ascending = False)
# print(importances)

Accuracy: 0.91013094417643
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.82      0.86      2377
           1       0.92      0.95      0.93      4878

    accuracy                           0.91      7255
   macro avg       0.91      0.89      0.90      7255
weighted avg       0.91      0.91      0.91      7255

Confusion Matrix:
[[1957  420]
 [ 232 4646]]
lead_time                               0.323159
avg_price_per_room                      0.160000
no_of_special_requests                  0.098108
arrival_date                            0.092518
arrival_month                           0.080359
market_segment_type                     0.054881
no_of_week_nights                       0.052004
no_of_weekend_nights                    0.037418
arrival_year                            0.025434
no_of_adults                            0.024662
type_of_meal_plan                       0.017853
room_type_reserved                     

In [ ]:
#######################
# LOGISTIC REGRESSION #
#######################

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

log_model = LogisticRegression(max_iter = 100, solver = "lbfgs")
log_model.fit(x_train, y_train)
y_pred = log_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# coef_df = pd.DataFrame({"Feature": x.columns, "Coefficient": log_model.coef_[0]}).sort_values(by = "Coefficient", ascending = False)
# print(coef_df)

Accuracy: 0.8126809097174362
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.63      0.69      2377
           1       0.83      0.90      0.87      4878

    accuracy                           0.81      7255
   macro avg       0.80      0.77      0.78      7255
weighted avg       0.81      0.81      0.81      7255

Confusion Matrix:
[[1499  878]
 [ 481 4397]]
                                 Feature  Coefficient
16                no_of_special_requests     1.076561
5             required_car_parking_space     0.275450
12                        repeated_guest     0.247875
6                     room_type_reserved     0.117797
9                          arrival_month     0.114139
14  no_of_previous_bookings_not_canceled     0.061963
0                           no_of_adults     0.000112
1                         no_of_children    -0.000073
10                          arrival_date    -0.025511
13          no_of_previous_cancellat

In [ ]:
##################
# MLP CLASSIFIER #
##################

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

mlp_model = MLPClassifier(hidden_layer_sizes = (64, 32), 
                          activation = "relu",
                          solver = "adam",
                          alpha = 0.0005,
                          learning_rate_init = 0.001,
                          max_iter = 100,
                          random_state = 42,
                          early_stopping = True,
                          n_iter_no_change = 5,
                          verbose = False)

mlp_model.fit(x_train, y_train)
y_pred = mlp_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8669882839421089
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.74      0.79      2377
           1       0.88      0.93      0.90      4878

    accuracy                           0.87      7255
   macro avg       0.86      0.83      0.84      7255
weighted avg       0.87      0.87      0.86      7255

Confusion Matrix:
[[1764  613]
 [ 352 4526]]


In [ ]:
############
# CATBOOST #
############

categorical_features = x.select_dtypes(include = ["object"]).columns.tolist()
print("Categorical features:", categorical_features)

cat_model = CatBoostClassifier(iterations = 500,
                               learning_rate = 0.05,
                               depth = 6,
                               loss_function = "Logloss",
                               eval_metric = "Accuracy",
                               random_seed = 42,
                               verbose = 100)

cat_model.fit(x_train,
              y_train,
              cat_features = categorical_features,
              eval_set = (x_test, y_test),
              use_best_model = True)

y_pred = cat_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Categorical features: []
0:	learn: 0.8168160	test: 0.8169538	best: 0.8169538 (0)	total: 106ms	remaining: 52.8s
100:	learn: 0.8597864	test: 0.8610613	best: 0.8625775 (99)	total: 1.07s	remaining: 4.23s
200:	learn: 0.8743625	test: 0.8758098	best: 0.8758098 (200)	total: 2.05s	remaining: 3.05s
300:	learn: 0.8847002	test: 0.8862853	best: 0.8864232 (290)	total: 3.03s	remaining: 2s
400:	learn: 0.8897312	test: 0.8894555	best: 0.8901447 (394)	total: 3.91s	remaining: 965ms
499:	learn: 0.8940041	test: 0.8915231	best: 0.8917988 (494)	total: 4.81s	remaining: 0us

bestTest = 0.8917987595
bestIteration = 494

Shrink model to first 495 iterations.
Accuracy: 0.8917987594762233
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.79      0.83      2377
           1       0.90      0.94      0.92      4878

    accuracy                           0.89      7255
   macro avg       0.88      0.87      0.87      7255
weighted avg       0.89      0.89    